# 05 — Writing your own engine

An engine is the first of the library's two extension points, and the claim the
design makes is strong: *"a new engine is a class with one method"*. A claim
about extensibility is only true if somebody has extended it **from the
outside**, so everything in this notebook is written against
`autosxtract.interfaces.Engine` and nothing else — no base class, no import of a
shipped engine.

The contracts are `typing.Protocol` and **structural on purpose**: an
implementation does not inherit from them, it merely has the methods. That is
what let `quality.stamp.Stamp` — written long before `interfaces.py` existed —
become a `StampStripper` without one line of it changing. Requiring inheritance
would have made that refactor a rewrite.

They are also `@runtime_checkable`, and that matters more than it looks. An
interface nobody checks is a comment, and this one had already drifted: the
`Engine` protocol declared `transcribe(pages, *, parallelism)` while `OCRStep`
had been passing `force_parallelism` for months. Nothing failed, because nothing
looked — and an engine written to the *published* contract would have crashed on
its first document.

In [ ]:
import logging

logging.disable(logging.INFO)

import pymupdf

from autosxtract import Cascade, Config, Engine
from autosxtract.image import dimensions
from autosxtract.steps import Context, OCRStep
from autosxtract.types import Line, Page, Transcription

# What our toy pretends to have read. Coordinates are FRACTIONS of the page, so
# the same script works at any rendering resolution.
SCRIPT = [
    ("EXCELENTISSIMO SENHOR DOUTOR JUIZ DE DIREITO DA VARA CIVEL", 0.96, (0.10, 0.07, 0.89, 0.09)),
    ("O requerente, nos autos do processo 0001234-56.2020.8.12.0001, vem requerer", 0.95, (0.10, 0.11, 0.89, 0.14)),
    ("a citacao do requerido conforme decisao proferida por esta vara civel em 17/03/2005.", 0.94, (0.10, 0.14, 0.89, 0.17)),
    ("PROTOCOLO 8A2F91C ASSINADO DIGITALMENTE", 0.71, (0.02, 0.17, 0.06, 0.80)),   # tall, narrow
    ("xzq lqp rn cl vv 1i", 0.31, (0.10, 0.51, 0.56, 0.54)),                        # junk
]
FLAT_TEXT = "\n".join(line for line, _, _ in SCRIPT)


def scanned_sheet() -> bytes:
    """Ink and no text layer, so the cascade has to reach the OCR step."""
    doc = pymupdf.open()
    page = doc.new_page()
    pix = pymupdf.Pixmap(pymupdf.csGRAY, pymupdf.IRect(0, 0, 400, 300))
    pix.set_rect(pix.irect, (210,))
    for y in range(20, 280, 24):
        pix.set_rect(pymupdf.IRect(20, y, 380, y + 6), (40,))
    page.insert_image(pymupdf.Rect(50, 50, 550, 750), stream=pix.tobytes("png"))
    data = doc.tobytes()
    doc.close()
    return data


document = scanned_sheet()
print(len(document), "bytes")

## The simple contract

`transcribe_page` returns `(text, confidence 0–100)` and `transcribe` returns the
whole document. Order is part of the contract: reassembling in completion order
scrambles the document, and **nothing downstream can tell that it did**.

The three members that answer `None` below are not stubs to be filled in later —
`None` is the honest reply from an engine without geometry, without a
detection-free path, or without an opinion about a witness reading, and it blocks
nothing.

Note what the toy does *not* do: it never claims a confidence that means
anything. Across 60 audited documents engine confidence did not separate a good
reading from an unsafe one — there was an unsafe document at 100 — so it enters
the pipeline as a floor against degenerate output and never as a quality
criterion (CLAUDE.md §7).

In [ ]:
class FlatEngine:
    """An engine written from the protocol alone: no base class, no import."""

    name = "toy_flat"
    #: Do threads add throughput here? ``False`` for a single hardware queue —
    #: the engine is the only party that knows, so it is the one that says.
    scales_with_threads = True

    def available(self) -> tuple[bool, str]:
        """``(can_run, reason)``. The reason is for logs, and it matters."""
        return True, "toy engine, always present"

    def transcribe_page(self, image: bytes) -> tuple[str, float]:
        return FLAT_TEXT, 93.0

    def read_page(self, image: bytes) -> Page | None:
        return None          # no geometry — honest, and it blocks nothing

    def recognize_crop(self, image: bytes) -> tuple[str, float] | None:
        return None          # no detection-free path

    def transcribe(self, pages, *, parallelism=4, force_parallelism=False):
        texts = [self.transcribe_page(image)[0] for image in pages]
        return Transcription(
            text="\n\n".join(texts),
            engine=self.name,
            pages_sent=len(pages),
            pages_answered=len(pages),   # answered < sent means a HOLE, not a short page
            mean_confidence=93.0,
        )

    def read_document(self, pdf_bytes, *, max_pages=3, min_reliable_words=3):
        return None          # "I don't know" — never "there is no text"


print("isinstance(FlatEngine(), Engine) ->", isinstance(FlatEngine(), Engine))

## The detailed contract, and what it is worth

`read_page` returns the same page **line by line**, each with a polygon and a
score. It is optional, and skipping it costs the containment layers — the
cheapest measured gain in the pipeline: entity recall 0.902 → 0.921, median CER
0.132 → 0.129, and p50 latency *falls* from 298 to 236 ms (§15). So implement it
whenever the backend exposes geometry.

Two details the toy below gets right on purpose, because both have bitten:

- **`Line.score` is always 0–1.** The layer thresholds depend on that scale; an
  engine reporting 0–100 must divide before building the `Line`, otherwise every
  line looks perfect.
- **The engine must actually look at the bytes it is handed.** Layer 2 re-reads
  *crops* through the same methods, so an engine that returns the same page
  whatever it is given will "recover" a whole page inside one line. The toy
  therefore reads the image header — which is exactly why the library ships
  `autosxtract.image.dimensions`: the detailed contract needs pixel sizes, and
  decoding a whole image to read two numbers would drag in Pillow or OpenCV,
  which the macOS install deliberately does not have.

In [ ]:
class GeometryEngine(FlatEngine):
    """The same toy, now answering the detailed half of the contract."""

    name = "toy_geometry"

    def read_page(self, image: bytes) -> Page | None:
        width, height = dimensions(image)
        if height < 800:
            # A Layer 2 crop, not a page. Answering ``None`` is what a real
            # engine does when it has nothing to detect in what it was given.
            return None
        return Page(
            width=width,
            height=height,
            lines=[
                Line(text, score, (
                    (x1 * width, y1 * height), (x2 * width, y1 * height),
                    (x2 * width, y2 * height), (x1 * width, y2 * height),
                ))
                for text, score, (x1, y1, x2, y2) in SCRIPT
            ],
        )

    def transcribe(self, pages, *, parallelism=4, force_parallelism=False):
        read = [self.read_page(image) for image in pages]
        if any(page is None for page in read):
            # ``Transcription.pages`` is filled only when EVERY page answered in
            # detail. A partial list would make the layers operate on a
            # different document from the one transcribed.
            return None
        return Transcription(
            text="\n\n".join(page.text for page in read),
            engine=self.name,
            pages_sent=len(pages),
            pages_answered=len(pages),
            mean_confidence=93.0,
            pages=read,
        )


print("isinstance(GeometryEngine(), Engine) ->", isinstance(GeometryEngine(), Engine))

### The same step, the same document, two engines

There is **no step per engine**: both go down the identical `OCRStep`. The only
difference in the provenance is the `layers` entry.

In [ ]:
for engine in (FlatEngine(), GeometryEngine()):
    result = OCRStep(engine).run(Context(pdf_bytes=document, config=Config()))
    print(f"{engine.name}")
    print(f"  accepted : {result.attempt.accepted}   chars: {result.attempt.chars}")
    print(f"  layers   : {result.attempt.details.get('layers')}")
    print(f"  text     : {result.candidate.text[-60:]!r}")
    print()

The flat engine's report is not empty — it says `skipped` and names the reason.
Skipping in silence is the antipattern the library fights: whoever reads the
provenance needs to know containment did **not** run, otherwise the absence of
`[illegible]` markers looks like a clean page.

The geometry engine's page, meanwhile, was classified line by line: the tall
narrow line was recognised as a vertical stamp by its *shape* before anything
tried to read it, and the junk line became one marker instead of polluting the
text.

## Registering it, and watching it descend the real cascade

`@register` puts the engine in the registry with a priority — lowest first — and
from there the cascade assembles itself. The shipped numbers come from
comparative measurement on the same 60-document sample:

```
10  Apple Vision   ~400 ms/page   92% of words, 100% of anchors
20  PP-OCRv6 tiny  ~500 ms/page   the off-Apple candidate
90  Tesseract      ~1.4 s/page    veto only; it does not persist text
```

Declaring `platforms` filters the engine out **before** it is instantiated, which
is what avoids importing what is not there.

In [ ]:
from autosxtract.engines import base as registry
from autosxtract.engines import diagnose, register

register(
    name="toy_geometry",
    priority=50,
    description="A toy engine, written for notebook 05",
)(GeometryEngine)

# Registering mutates a process-wide registry — fine in a notebook, worth knowing
# before you do it inside a library import.
for name, ok, reason in diagnose():
    print(f"  [{'x' if ok else ' '}] {name:<14} {reason}")

In [ ]:
cascade = Cascade(Config(engines=["toy_geometry"]))
print("assembled cascade:", cascade.names)
print()

result = cascade.extract(document, identifier="toy.pdf")
print(result.provenance)
print()
for key, value in result.to_dict().items():
    if key != "text":
        print(f"  {key:<12} {value}")

`Config(engines=[...])` is the operator's explicit order, and it **wins even when
it names an engine that cannot run** — the step becomes a refused attempt with the
reason attached rather than an error. Silencing the operator's choice would be
worse than honouring it.

## What breaks when a method is missing

`isinstance` against a runtime-checkable Protocol checks for the *presence* of
the members, not their signatures. It is a cheap, real check — and the cell below
shows both what it catches and what it does not.

In [ ]:
class Incomplete:
    """Everything a naive first draft has, and nothing else."""

    name = "incomplete"
    scales_with_threads = True

    def available(self):
        return True, "ok"

    def transcribe_page(self, image):
        return "texto lido pela engine incompleta", 90.0


print("isinstance(Incomplete(), Engine) ->", isinstance(Incomplete(), Engine))

required = [m for m in dir(Engine) if not m.startswith("_")]
print("missing members:", [m for m in required if not hasattr(Incomplete, m)])
print()

try:
    OCRStep(Incomplete()).run(Context(pdf_bytes=document, config=Config()))
except AttributeError as exc:
    print("running it anyway ->", type(exc).__name__, ":", exc)

That `AttributeError` arrives at the *call site*, on somebody's document, in the
middle of a batch — which is the whole argument for the `isinstance` check and for
`tests/contract/test_interfaces.py`, where every shipped engine is asserted against the
protocol **and** against its signatures. Presence is not enough: a method that
exists with the wrong keyword arguments passes `isinstance` and fails on the first
document, which is precisely the drift that went unnoticed for months.

## An engine that says no

The second rule of the contract: **a missing engine is never an exception.**
`available()` returns `(False, reason)` in words, the step goes inert, and the
cascade moves on. The absence of a tool is not evidence about the document —
treating "I have no OCR" as "the page is empty" switches the pipeline off in
silence (§3).

In [ ]:
class Unavailable(FlatEngine):
    name = "toy_absent"

    def available(self):
        return False, "toy_absent unavailable: pretend the weights are missing"


refused = OCRStep(Unavailable()).run(Context(pdf_bytes=document, config=Config()))
print("accepted :", refused.attempt.accepted)
print("reason   :", refused.attempt.reason)
print("candidate:", refused.candidate)
print()

# And inside a cascade: an inert step, a reason in the provenance, no exception.
mixed = Cascade(Config(), steps=[OCRStep(Unavailable()), OCRStep(GeometryEngine())])
print(mixed.extract(document, "toy.pdf").provenance)

## Checklist for a real engine

Everything above is toy; these are the parts that are not.

1. **Confidence 0–100 from `transcribe_page`, 0–1 on every `Line`.** Two scales,
   both load-bearing, and mixing them makes every line look perfect.
2. **Preserve page order in `transcribe`.** Reassembling in completion order
   scrambles the document invisibly.
3. **Fill `Transcription.pages` only when *every* page answered in detail**, and
   report `pages_answered < pages_sent` honestly — a hole passes every volume
   test and disappears without a trace.
4. **Use `failures` for pages that *raised*.** A page read as blank and a page
   never reached look identical in the text, and only one of them means the
   engine is down. That resemblance is how a dead worker once cost 488 documents
   and 28,239 characters with nobody noticing.
5. **Never reuse the main OCR instance to recognise a crop** (§16). Calling
   `rapidocr` with `use_det=False` turns detection off *permanently on that
   object*: measured, the next whole-page read returned 1 line where it had
   returned 56, and the document came out with 1 character instead of 3,900. The
   defect is silent, order-dependent, and invisible before the second page of a
   batch. `PaddleEngine._recognizer()` therefore keeps a separate instance. The
   general rule: a third-party library with per-instance state is a shared
   resource, and any path that changes its configuration needs its own object.
6. **Say `scales_with_threads = False` if a single hardware queue sits behind
   you.** Apple's Neural Engine served one request at a time while latency went
   from 430 ms to 3,492 ms across 1 to 12 threads at constant throughput. Only
   the engine knows this — but the operator can still override it through
   `Config.engine_parallelism`, because that declaration was measured on one
   machine.

Next: **06 — writing your own step**, the second extension point, and what
`expensive = True` sets in motion.